# Моделирование датчика Шака–Гартмана

Цель расчёта — получить распределение интенсивности в фокальной плоскости микролинзового массива для пучка с заданным фазовым искажением.

## Постановка расчёта

Моделируется регистрация гауссова пучка датчиком Шака–Гартмана. Фаза входного поля задаётся случайной линейной комбинацией полиномов Цернике.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from AO import ShackHartmann, GaussianBeam, ApertureFunctions

In [ ]:
N = 4096
dx = 1e-4
wavelength = 650e-9
L_screen = N * dx

zernike_coefficients = np.random.uniform(0, 1, 6)
phase_0 = ApertureFunctions.zernike_image(zernike_coefficients, N)

beam = GaussianBeam(N=N, size_screen=L_screen, w0=L_screen / 2)
E = beam.E * np.exp(1j * phase_0)

## Модель микролинзового массива

Датчик задаётся числом субапертур, их периодом, радиусом, фокусным расстоянием и длиной волны.

In [ ]:
sensor = ShackHartmann(
    size=N,
    num_subapertures=15,
    period=300e-6,
    radius_subapertures=136e-6,
    focal_length=0.00322,
    wavelength=wavelength,
    mode="radial",
)

E_focal = sensor.E_focal(E * sensor.aperture_mask)
intensity_input = np.abs(E) ** 2
intensity_focal = np.abs(E_focal) ** 2

## Результат регистрации

Сравниваются начальная фаза, интенсивность входного пучка, фазовая маска датчика и распределение интенсивности в фокальной плоскости.

In [ ]:
I_min = min(intensity_input.min(), intensity_focal.min())
I_max = max(intensity_input.max(), intensity_focal.max())

fig, axes = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)

im = axes[0, 0].imshow(phase_0 / (2 * np.pi), cmap="hsv")
axes[0, 0].set_title("Начальная фаза, $\varphi / 2\pi$")
fig.colorbar(im, ax=axes[0, 0])

im = axes[0, 1].imshow(intensity_input, cmap="hot")
axes[0, 1].set_title("Начальная интенсивность")
fig.colorbar(im, ax=axes[0, 1])

im = axes[1, 0].imshow(sensor.phase_mask)
axes[1, 0].set_title("Фазовая маска датчика")
fig.colorbar(im, ax=axes[1, 0])

im = axes[1, 1].imshow(intensity_focal, cmap="hot")
im.set_clim(I_min, I_max)
axes[1, 1].set_title("Интенсивность в фокальной плоскости")
fig.colorbar(im, ax=axes[1, 1])

plt.show()